In [55]:
import numpy as np
import pandas as pd

from power_grid_model import (
    AttributeType,
    CalculationMethod,
    CalculationType,
    ComponentAttributeFilterOptions,
    ComponentType,
    DatasetType,
    LoadGenType,
    PowerGridModel,
    attribute_dtype,
    initialize_array,
)


In [ ]:
df_topology = pd.read_csv("Practicum_1_Python/grid_topology.csv")

node_ids = np.unique(
    df_topology[["FROM", "TO"]].to_numpy().flatten()
)

#print(node_ids)


nodes = initialize_array(DatasetType.input, ComponentType.node, len(node_ids))

nodes[AttributeType.id] = node_ids
nodes[AttributeType.u_rated] = 400*np.ones(np.size(nodes))

line = initialize_array(DatasetType.input, ComponentType.line, len(df_topology))

line[AttributeType.id] = df_topology.index + np.max(node_ids)

line[AttributeType.from_node] = df_topology["FROM"]
line[AttributeType.from_node] = df_topology["TO"]
line[AttributeType.from_status] = np.ones(len(df_topology))
line[AttributeType.to_status] = np.ones(len(df_topology))
line[AttributeType.r1] = df_topology["Raa (Ω)"]
line[AttributeType.x1] = df_topology["Xaa (Ω)"]
line[AttributeType.i_n] = df_topology["Imax"]
line[AttributeType.c1] = 10e-6*np.ones(len(df_topology))
line[AttributeType.tan1] = 0*np.ones(len(df_topology))

line_ids = line[AttributeType.id]

df_loads  = pd.read_csv("Practicum_1_Python/consumption_one_year_15min.csv")
df_source = pd.read_csv("Practicum_1_Python/pv_one_year_15min.csv")

timestamps_load = df_loads["date"]
columns_loads = list(df_loads)
#print(df_loads.columns[1:].str.extract(r'(\d+)', expand=False).astype(int))
#print(line_ids)
sym_load = initialize_array(DatasetType.input, ComponentType.sym_load, len(df_loads.columns)-1)
sym_load[AttributeType.id] = np.arange(1,len(df_loads.columns)) + np.max(line_ids)

sym_load_ids = sym_load[AttributeType.id]
#print(sym_load_ids)

sym_loads_over_time = []
sources_over_time   = []

powerfactor=0.95
#print(df_loads.iloc[1,1:])

def calc_reactive_Power(P, powerFactor = 0.95):\
    return -P * np.tan(np.arccos(powerFactor))

for i in np.arange(len(df_loads[0:10])):
    #loads
    sym_load = initialize_array(DatasetType.input, ComponentType.sym_load, len(df_loads.columns)-1)
    sym_load[AttributeType.id]          = np.arange(1,len(df_loads.columns)) + np.max(line_ids)
    sym_load[AttributeType.node]        = df_loads.columns[1:].str.extract(r'(\d+)', expand=False).astype(int)
    sym_load[AttributeType.status]      = np.ones(len(df_loads.columns)-1)
    sym_load[AttributeType.type]        = [LoadGenType.const_power] * len(sym_load)
    sym_load[AttributeType.p_specified] = df_loads.iloc[i,1:] * 1000
    sym_load[AttributeType.q_specified] = calc_reactive_Power(sym_load["p_specified"])
    sym_loads_over_time.append(sym_load)

    #sources (PV)
    source = initialize_array(DatasetType.input, ComponentType.source, len(df_source.columns)-1)
    source[AttributeType.id]     = np.arange(1,len(df_source.columns)) + np.max(sym_load["id"])
    source[AttributeType.node]   = df_source.columns[1:].str.extract(r'(\d+)', expand=False).astype(int)
    source[AttributeType.status] = np.ones(len(df_source.columns)-1)
    source[AttributeType.u_ref]  = np.ones(len(df_source.columns)-1)
